# Solution 03 on Google Colab — VLM LoRA fine-tune, synthetic augmented (no LLM-assisted injection)

Runs the existing `solutions/03_vlm_lora_synthetic_augmented/run_pipeline.py` unchanged against **Qwen/Qwen3-VL-8B-Instruct** (the script's own default, same as solution 02). This notebook only handles Colab-specific orchestration: mounting Drive, copying the project locally, installing dependencies, running the pipeline, and syncing results back.

**Before running anything:** `Runtime > Change runtime type > T4/L4/A100 GPU`.

**Assumption about your Drive layout:** same as solution 02 -- `solutions/`, `participant_kit/`, `data/`, `shroom-vis-images/` all live together under one folder on your Drive. Set `DRIVE_PROJECT_PATH` below.

**Two things specific to this solution:**
1. **The EasyOCR pass over ~2,054 images is CPU-bound and slow (measured at ~6 hours for a full pass locally).** If you already have an `ocr_cache.json` on Drive from a previous run (including one copied over from solution 04, which uses an identical OCR step), this notebook detects and reuses it automatically, skipping that step entirely. Otherwise the first full run builds it once and saves it back to Drive immediately (before training even starts) so you never pay that cost twice.
2. **Method (c), LLM-assisted injection, is never used in this solution.** `run_pipeline.py` in this folder always passes `--skip_llm` to `generate_synthetic.py`, with no flag to turn it back on -- this folder exists specifically to isolate methods (a)+(b) alone at r_max=8, the one cell of the rarity-cap x LLM-assisted-injection ablation the original shared-task submission never trained. See [solution 04](../04_vlm_lora_synthetic_augmented_llm/colab_run_solution04.ipynb) for the identical pipeline with LLM-assisted injection included by default.

**Checkpointing and resume (fixed after a real incident: a full run OOMed Colab's ~24h session limit at 95% complete with nothing saved, because training only ever saved once, at the very end, to local disk that got wiped with the session).** The full-run cell now:
- Points `--checkpoint_dir` at a path under your mounted Drive (`CHECKPOINT_DIR`, set in the config cell below), not local `/content` disk, so a saved checkpoint survives the session dying.
- Passes `--save_steps` (default 250 optimizer steps, ~40-45 min at observed A100 throughput) so the adapter is saved periodically during training, not just once at the end.
- Passes `--resume`, so re-running the exact same full-run cell after a cutoff continues from the last save instead of starting over (data order is deterministic given the same `--seed`, so it correctly skips only the examples already processed).
- The smoke test intentionally does **not** use `--checkpoint_dir`/`--resume` -- it saves to a separate local-only checkpoint dir, so a smoke test can never overwrite real full-run progress on Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# --- Config: edit these ---
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/SHROOM"   # <-- change to wherever your project actually lives on Drive
LOCAL_PROJECT_PATH = "/content/SHROOM"
GPU_INDEX = 0        # Colab only ever exposes one GPU -- leave this at 0
EPOCHS = 2
GRAD_ACCUM = 8
SAVE_STEPS = 250     # full-run checkpoint frequency (optimizer steps) -- worst-case progress lost
                      # to a session cutoff is about one save interval, not the whole run

sol03_local = f"{LOCAL_PROJECT_PATH}/solutions/03_vlm_lora_synthetic_augmented"  # defined here (not
                                                                                  # only in the OCR-cache
                                                                                  # cell below) so a
                                                                                  # resume-for-test-
                                                                                  # prediction-only session
                                                                                  # (new runtime, skipping
                                                                                  # straight past smoke
                                                                                  # test/full run to the
                                                                                  # test-prediction section)
                                                                                  # doesn't need to run
                                                                                  # training-only cells
                                                                                  # just to get this path.

# The full run's checkpoint lives on Drive (not local disk) specifically so it survives a session
# being killed mid-training -- see the checkpointing/resume note above. The smoke test deliberately
# does NOT use this path (see its cell below) so it can never overwrite real full-run progress.
CHECKPOINT_DIR = f"{DRIVE_PROJECT_PATH}/solutions/03_vlm_lora_synthetic_augmented/checkpoint"

# No SKIP_LLM toggle and no ANTHROPIC_API_KEY handling here -- this solution's run_pipeline.py
# always passes --skip_llm, so method (c) never runs from this folder regardless. See solution 04
# if you want LLM-assisted injection.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("DRIVE_PROJECT_PATH:", DRIVE_PROJECT_PATH)
print("LOCAL_PROJECT_PATH:", LOCAL_PROJECT_PATH)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)

In [ ]:
# Copy just the subfolders the pipeline actually needs from Drive to local disk.
# Skips anything already copied (and non-empty), so re-running this cell after a runtime
# restart is cheap. Fails loudly (instead of silently) if a source folder is missing on
# Drive or a copy comes out empty -- catches a wrong DRIVE_PROJECT_PATH / folder-name
# mismatch here, instead of it surfacing 10 minutes later as "0 usable conversations"
# deep inside training.
os.makedirs(LOCAL_PROJECT_PATH, exist_ok=True)
for sub in ["data", "shroom-vis-images", "participant_kit", "solutions"]:
    src = f"{DRIVE_PROJECT_PATH}/{sub}"
    dst = f"{LOCAL_PROJECT_PATH}/{sub}"
    if os.path.exists(dst) and os.listdir(dst):
        print(f"skip (already present locally, {len(os.listdir(dst))} entries): {sub}")
        continue
    if not os.path.exists(src):
        raise FileNotFoundError(
            f"{src} does not exist on your Drive. Check DRIVE_PROJECT_PATH and the exact folder "
            f"name/casing on your Drive (run `!ls \"{DRIVE_PROJECT_PATH}\"` in a new cell to see "
            f"what's actually there)."
        )
    print(f"copying {sub} from Drive ... (this can take a few minutes for shroom-vis-images/)")
    !cp -r "{src}" "{dst}"
    n = len(os.listdir(dst)) if os.path.exists(dst) else 0
    if n == 0:
        raise RuntimeError(
            f"Copy of {sub} produced an empty directory at {dst} -- something went wrong "
            f"(check the cp output above for errors, e.g. permission issues or a Drive timeout)."
        )
    print(f"  -> {n} entries copied")
print("done")
!ls "{LOCAL_PROJECT_PATH}"

In [ ]:
!pip install -q "git+https://github.com/huggingface/transformers"
!pip install -q "huggingface-hub<1.0" "accelerate>=0.34" "peft>=0.13" sentencepiece
!pip install -q bitsandbytes easyocr sacremoses

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

major, minor = (int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
if (major, minor) < (2, 6):
    print("\ntorch is older than 2.6 -- upgrading (see solution 02's notebook for why):")
    !pip install -q -U torch torchvision torchaudio
    print("IMPORTANT: Runtime > Restart runtime now, then re-run this notebook from the top.")
else:
    print("torch version OK.")

In [ ]:
%cd {LOCAL_PROJECT_PATH}
!python -m solutions.common.scorer_local

In [ ]:
# Reuse an existing OCR cache from Drive if a previous run already built one --
# avoids repeating the ~6hr CPU-bound EasyOCR pass. Also checks solution 04's cache
# (same OCR step, identical images) since either solution having already built one
# means neither needs to again.
import shutil

ocr_cache_drive = f"{DRIVE_PROJECT_PATH}/solutions/03_vlm_lora_synthetic_augmented/ocr_cache.json"
ocr_cache_drive_sol04 = f"{DRIVE_PROJECT_PATH}/solutions/04_vlm_lora_synthetic_augmented_llm/ocr_cache.json"
ocr_cache_local = f"{sol03_local}/ocr_cache.json"

SKIP_OCR_EXTRACT = False
if os.path.exists(ocr_cache_drive):
    print("Found an existing ocr_cache.json on Drive (solution 03) -- reusing it, skipping the EasyOCR pass.")
    os.makedirs(sol03_local, exist_ok=True)
    shutil.copy(ocr_cache_drive, ocr_cache_local)
    SKIP_OCR_EXTRACT = True
elif os.path.exists(ocr_cache_drive_sol04):
    print("Found an existing ocr_cache.json from solution 04 on Drive -- reusing it (same OCR step).")
    os.makedirs(sol03_local, exist_ok=True)
    shutil.copy(ocr_cache_drive_sol04, ocr_cache_local)
    SKIP_OCR_EXTRACT = True
else:
    print("No existing ocr_cache.json on Drive -- this run will build one from scratch (slow, budget hours).")
print("SKIP_OCR_EXTRACT =", SKIP_OCR_EXTRACT)

Smoke test next. Deliberately does **not** pass `--skip_ocr_extract` unless we already found a cache to reuse above -- on a genuinely first run, this exercises the real EasyOCR path on `--smoke_test`'s capped 15-image sample first, rather than discovering an EasyOCR bug for the first time during the full multi-hour pass (an earlier version of this project's runbook got this ordering wrong; see solution 03's README).

In [ ]:
%cd {sol03_local}
skip_ocr_flag = "--skip_ocr_extract" if SKIP_OCR_EXTRACT else "

In [ ]:
!python run_pipeline.py --smoke_test --gpu_index {GPU_INDEX} {skip_ocr_flag}

In [ ]:
# The smoke test above -- if it ran the real EasyOCR path (SKIP_OCR_EXTRACT was False) -- just
# wrote a 15-image-only ocr_cache.json locally. generate_synthetic.py reuses whatever cache file
# already exists at this path with no size/completeness check, so if we don't clear it here, the
# full run below would silently reuse that 15-image cache instead of building a real one covering
# all ~2,054 training images -- no error, just far fewer OCR-grounded synthetic examples than
# intended. Only delete it if it was NOT the real cache reused from Drive.
if not SKIP_OCR_EXTRACT and os.path.exists(ocr_cache_local):
    os.remove(ocr_cache_local)
    print("Removed the smoke-test-scoped ocr_cache.json -- the full run below will build a real one.")
else:
    print("Nothing to clean up (either no cache was built yet, or it's the real one reused from Drive).")

Once the smoke test finishes cleanly, run the full pipeline. This is the multi-hour step (data generation + training + prediction).

**Sequence-length OOM protection**: `train_lora.py` skips (not crashes on) any example whose token
sequence exceeds `--max_seq_len` (default 4096) -- guards against a rare pathologically long example
(most likely from `inject_repetition_loop`, method b's 'other' class) spiking attention memory and
OOMing an otherwise-healthy run. On by default, no flag needed.

**Session-cutoff protection**: the cell below passes `--checkpoint_dir "{CHECKPOINT_DIR}"` (Drive, not
local disk), `--save_steps {SAVE_STEPS}`, and `--resume`. If Colab kills the session partway through
(hit in practice: a run OOMed the ~24h limit at 95% complete with the old code, losing everything since
it only saved once, at the very end, to local disk that gets wiped) -- just reconnect a new session, run
this notebook from the top again (mount Drive, config, copy-from-Drive, install deps), and re-run this
exact same full-run cell. `--resume` will find the last Drive-saved checkpoint and continue from there,
losing at most one `--save_steps` interval of progress instead of the entire run.

OOM fallback for a smaller GPU (T4/L4 instead of A100), same as solution 02:
```
!python run_pipeline.py --gpu_index {GPU_INDEX} --epochs {EPOCHS} --grad_accum {GRAD_ACCUM} {skip_ocr_flag} --checkpoint_dir "{CHECKPOINT_DIR}" --save_steps {SAVE_STEPS} --resume --max_pixels 401408 --target_modules q_proj,k_proj,v_proj,o_proj
```

In [ ]:
!python run_pipeline.py --gpu_index {GPU_INDEX} --epochs {EPOCHS} --grad_accum {GRAD_ACCUM} {skip_ocr_flag} --checkpoint_dir "{CHECKPOINT_DIR}" --save_steps {SAVE_STEPS} --resume

In [ ]:
# Save the freshly-built OCR cache back to Drive right away (even before the final results
# copy below), so a later run/disconnect never has to repeat the slow EasyOCR pass again.
if os.path.exists(ocr_cache_local):
    os.makedirs(os.path.dirname(ocr_cache_drive), exist_ok=True)
    !cp "{ocr_cache_local}" "{ocr_cache_drive}"
    print("OCR cache saved to Drive.")
else:
    print("No local ocr_cache.json found (unexpected) -- nothing to save.")

In [ ]:
# Copy the rest of the results back to Drive: predictions/scores, plus the augmented training set
# for provenance/debugging. No LLM cache to copy -- this solution never calls method (c).
# checkpoint/ is NOT copied here -- the full-run cell above already saved it directly to
# CHECKPOINT_DIR on Drive throughout training.
dest = f"{DRIVE_PROJECT_PATH}/solutions/03_vlm_lora_synthetic_augmented"
os.makedirs(dest, exist_ok=True)
for item in ["val_predictions.jsonl", "val_scores.json", "train_augmented.jsonl"]:
    src = f"{sol03_local}/{item}"
    if os.path.exists(src):
        print(f"copying {item} -> Drive ...")
        !cp -r "{src}" "{dest}/"
    else:
        print(f"skip (not found): {item}")
print("done -- check", dest, "in your Drive")

In [ ]:
import json
with open(f"{sol03_local}/val_scores.json") as f:
    print(json.dumps(json.load(f), indent=2))

## Test-set predictions (for SHROOM website submission)

Everything above only ever predicted on `data/val.jsonl` -- deliberately, so you could inspect val
scores before committing to a submission. This section runs the trained checkpoint over the real,
**unlabeled** `data/test.jsonl` instead, producing the actual file you attach to the SHROOM submission
form. There is no scoring step here -- `test.jsonl` has no gold labels, so there's nothing to compare
predictions against locally.

This calls solution 02's `predict.py` directly (not through `run_pipeline.py`, which is hardcoded to
`val.jsonl` and would also try to score against labels that don't exist for test data), pointed at
`CHECKPOINT_DIR` -- the same Drive path the full-run cell trained into, so this also works as a
**standalone resume-for-prediction-only session**: mount Drive, run the config cell, then skip straight
here without re-running training at all.

**If you used the OOM fallback flags** (`--max_pixels 401408 --target_modules ...`) for the full
training run above, add the same `--max_pixels 401408` here too -- `predict.py`'s docstring notes this
must match what was used at training time. `--target_modules` itself doesn't need repeating here (the
checkpoint's adapter config already encodes which modules got LoRA-tuned).

In [ ]:
test_predictions_path = f"{sol03_local}/test_predictions.jsonl"

predict_script = f"{LOCAL_PROJECT_PATH}/solutions/02_vlm_lora_real_data/predict.py"
test_file_path = f"{LOCAL_PROJECT_PATH}/data/test.jsonl"

!python "{predict_script}" --checkpoint "{CHECKPOINT_DIR}" --val_file "{test_file_path}" --output "{test_predictions_path}" --gpu_index {GPU_INDEX}

In [ ]:
# Copy the test-set predictions to Drive -- this is the file to upload to the SHROOM submission form.
dest = f"{DRIVE_PROJECT_PATH}/solutions/03_vlm_lora_synthetic_augmented"
os.makedirs(dest, exist_ok=True)
if os.path.exists(test_predictions_path):
    !cp "{test_predictions_path}" "{dest}/"
    print(f"copied test_predictions.jsonl -> {dest}/")
else:
    print("test_predictions.jsonl not found -- the predict.py cell above must not have completed.")

In [ ]:
# Quick sanity peek: line count + first record, so you can eyeball the format before submitting.
import json

with open(test_predictions_path) as f:
    lines = f.readlines()
print(f"{len(lines)} predictions written")
print(json.dumps(json.loads(lines[0]), indent=2, ensure_ascii=False))